In [0]:
from pyspark.sql.functions import col, regexp_extract, expr


df = spark.table("assignment_catalog.bronze_layer.sales_raw")
df.filter(~col("date").rlike(r"\d{4}-\d{2}-\d{2}")).display(truncate=False)

In [0]:
df_clean = df.select(
    expr("try_cast(date as date)").alias("date"),
    
    col("item_id").alias("sku"),
    
    expr("try_cast(qty_sold as double)").cast("int").alias("units"),
    
    (
        expr("try_cast(qty_sold as double)") *
        expr("try_cast(mrp as double)")
    ).alias("revenue"),
    
    regexp_extract("source_file", r'.*-(\w+)-Sales\.csv$', 1).alias("data_source")
)

df_clean = df_clean.dropna()


In [0]:
df_clean.display()
df_clean.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("assignment_catalog.silver_layer.sales_clean")

In [0]:
df.display(5)
df.printSchema()